In [ ]:
# ==============================================================================
# ERP DATA MIGRATION PIPELINE
# Role: Data Analyst 
# Description: Extracts hidden attributes via Regex, unifies categories, 
# and builds a strict Parent/Child matrix for Odoo ERP importing.
# ==============================================================================

import pandas as pd
import numpy as np
import re

print("Initializing Data Pipeline...")

# 1. LOAD RAW DATA
# ------------------------------------------------------------------------------
# Note: Using a sample dataset for public repository. 
# Original script processed 5,725+ rows of legacy inventory data.
df = pd.read_excel('sample_raw_data.xlsx', engine='openpyxl')

# Helper to dynamically find column names regardless of capitalization
def get_col(options):
    for opt in options:
        if opt in df.columns: return opt
    return None

COL_ITEM_NAME = get_col(['ITEM NAME', 'Item Name'])
COL_BARCODE = get_col(['BARCODE VALUE', 'Barcode Value'])
COL_COST = get_col(['STK VALUE ON PURPRICE', 'Stk Value On PurPrice'])
COL_SELL = get_col(['STK VALUE ON SELLING', 'Stk Value On Selling'])
COL_CAT2 = get_col(['CAT 2', 'Cat 2'])
COL_CAT3 = get_col(['CAT 3', 'Cat 3'])
COL_SIZE = get_col(['SIZE', 'Size', 'SIZE (Values of Attribute 2)'])
COL_COLOR = get_col(['COLOR', 'Color', 'COLOUR (Values of Attribute 1)'])


# 2. DICTIONARIES & REGEX PATTERNS
# ------------------------------------------------------------------------------
COLOR_SYNONYMS = {
    'MIX': 'Multi', 'MIXED': 'Multi', 'MIX COLOR': 'Multi', 'MULTI': 'Multi', 'ASST': 'Assorted',
    'GRAY': 'Grey', 'BROWN/COFFEE': 'Brown', 'COFFEE COLOR': 'Coffee', 'BLCK+RED': 'Black/Red'
}

KNOWN_COLORS = ['RED', 'BLUE', 'GREEN', 'YELLOW', 'BLACK', 'WHITE', 'ORANGE', 'PINK', 
                'PURPLE', 'GREY', 'BROWN', 'BEIGE', 'SAND', 'TAN', 'GOLD', 'SILVER']

SIZE_SYNONYMS = {'SMALL': 'S', 'MEDIUM': 'M', 'LARGE': 'L', 'EXTRA SMALL': 'XS', 'EXTRA LARGE': 'XL'}
KNOWN_SIZES = ['XS', 'S', 'M', 'L', 'XL', 'XXL', '2XL', '3XL']

def force_spacing(val, unit_regex):
    """Normalizes metric spacing (e.g., '1.5KG' -> '1.5 KG')"""
    if pd.isna(val) or not val: return val
    val = str(val).upper().replace(',', '.') 
    val = re.sub(r'(\d+(?:\.\d+)?)\s*' + unit_regex, r'\1 \2', val)
    return re.sub(r'\s+', ' ', val).strip()

def clean_dimension(val):
    """Standardizes dimensions (e.g., '25,8X25,8X5Cm' -> '25.8 X 25.8 X 5.0 CM')"""
    val = str(val).upper().replace(',', '.')
    val = val.replace('*', ' X ').replace('X', ' X ')
    val = re.sub(r'\s+', ' ', val)
    val = re.sub(r'(\d)\s*(CM|MM|INCH|IN)', r'\1 \2', val)
    return val.strip()

def parse_string_to_attributes(text):
    """The 'Sledgehammer': Rips specific metrics out of chaotic free-text strings."""
    if not text or text == 'NAN' or text == 'NONE':
        return "", "", "", "", ""
        
    text = str(text).upper().strip()
    c, s, w, cap, d = "", "", "", "", ""

    # Extract Dimensions
    dim_match = re.search(r'(\d+(?:\.\d+)?\s*(?:X|\*)\s*\d+(?:\.\d+)?(?:\s*(?:X|\*)\s*\d+(?:\.\d+)?)?\s*(?:CM|MM|INCH|IN)?)', text)
    if dim_match:
        d = clean_dimension(dim_match.group(1))
        text = text.replace(dim_match.group(1), '') 

    # Extract Capacity
    cap_match = re.search(r'(\d+(?:\.\d+)?\s*(ML|LITER|LITRE|L|GAL)\b)', text)
    if cap_match:
        cap = force_spacing(cap_match.group(1), r'(ML|LITER|LITRE|L|GAL)')
        text = text.replace(cap_match.group(1), '')

    # Extract Weight
    weight_match = re.search(r'(\d+(?:\.\d+)?\s*(KG|G|LBS|LB|OZ)\b)', text)
    if weight_match:
        w = force_spacing(weight_match.group(1), r'(KG|G|LBS|LB|OZ)')
        text = text.replace(weight_match.group(1), '')

    # Extract Size
    for syn, clean_s in SIZE_SYNONYMS.items():
        if re.search(rf'\b{syn}\b', text):
            s = clean_s
            text = text.replace(syn, '')
            break
    if not s:
        for size in sorted(KNOWN_SIZES, key=len, reverse=True): 
            if re.search(rf'\b{size}\b', text):
                s = size
                break
    if not s:
        cm_match = re.search(r'\b(\d+\s*CM)\b', text)
        if cm_match:
            s = force_spacing(cm_match.group(1), r'(CM)')

    # Extract Color
    for syn, clean_c in COLOR_SYNONYMS.items():
        if syn in text:
            c = clean_c
            break
    if not c:
        found_colors = [color for color in KNOWN_COLORS if color in text]
        if found_colors:
            c = "/".join([col.title() for col in found_colors])

    return s, c, w, cap, d


# 3. APPLY SMART SORTING
# ------------------------------------------------------------------------------
print("Extracting and Unifying Attributes...")
def smart_sort(row):
    raw_size = str(row.get(COL_SIZE, '')).upper()
    raw_color = str(row.get(COL_COLOR, '')).upper()
    item_name = str(row.get(COL_ITEM_NAME, '')).upper()
    
    s1, c1, w1, cap1, d1 = parse_string_to_attributes(raw_size)
    s2, c2, w2, cap2, d2 = parse_string_to_attributes(raw_color)
    
    final_size = s1 or s2
    final_color = c1 or c2
    final_weight = w1 or w2
    final_capacity = cap1 or cap2
    final_dimension = d1 or d2
    
    # Fallback to scanning Item Name if attributes are still missing
    if not final_weight:
        _, _, final_weight, _, _ = parse_string_to_attributes(item_name)
    if not final_capacity:
        _, _, _, final_capacity, _ = parse_string_to_attributes(item_name)
    if not final_dimension:
        _, _, _, _, final_dimension = parse_string_to_attributes(item_name)

    return pd.Series([final_size, final_color, final_weight, final_capacity, final_dimension])

df[['Unified_Size', 'Unified_Color', 'Unified_Weight', 'Unified_Capacity', 'Unified_Dimension']] = df.apply(smart_sort, axis=1)
for col in ['Unified_Size', 'Unified_Color', 'Unified_Weight', 'Unified_Capacity', 'Unified_Dimension']:
    df[col] = df[col].replace('', np.nan)


# 4. BUILD ODOO PARENT/CHILD MATRIX
# ------------------------------------------------------------------------------
print("Building Odoo ERP Hierarchy...")
final_odoo_rows = []

odoo_columns = [
    'Name', 'Attributes', 'Values', 'Barcode', 'Description', 'Can be sold', 
    'Can be purchased', 'Available in POS', 'Product Type', 'Tracking', 
    'Product Category', 'Cost', 'Sales Price', 'Invoicing policy'
]

if COL_ITEM_NAME:
    for name, group in df.groupby(COL_ITEM_NAME, dropna=False):
        if pd.isna(name): continue
            
        # Determine the active variant category (Size, Color, etc.)
        attr_map = {
            'SIZE': group['Unified_Size'].dropna().unique(),
            'COLOR': group['Unified_Color'].dropna().unique(),
            'WEIGHT': group['Unified_Weight'].dropna().unique(),
            'CAPACITY': group['Unified_Capacity'].dropna().unique(),
            'DIMENSION': group['Unified_Dimension'].dropna().unique()
        }
        
        active_attrs = {k: v for k, v in attr_map.items() if len(v) > 0}
        attr_name = ''
        attr_vals = ''
        
        if len(active_attrs) > 0:
            attr_name = list(active_attrs.keys())[0] 
            attr_vals = ",".join(sorted(active_attrs[attr_name]))
            
        is_matrix = len(active_attrs) > 0 and len(group) > 1
        first_row = True 
        
        for index, row in group.iterrows():
            odoo_row = {col: '' for col in odoo_columns} 
            
            # Variant Specific Data (Prints on every row)
            odoo_row['Barcode'] = row.get(COL_BARCODE, '')
            odoo_row['Cost'] = row.get(COL_COST, '')
            odoo_row['Sales Price'] = row.get(COL_SELL, '')

            # Parent Product Data (Only prints on the first row to create the Odoo hierarchy)
            if (is_matrix and first_row) or not is_matrix:
                odoo_row['Name'] = name
                odoo_row['Attributes'] = attr_name if is_matrix else ''
                odoo_row['Values'] = attr_vals if is_matrix else ''
                
                # Global ERP Rules
                odoo_row['Description'] = row.get(COL_CAT3, '') 
                odoo_row['Can be sold'] = 'TRUE'
                odoo_row['Can be purchased'] = 'TRUE'
                odoo_row['Available in POS'] = 'TRUE'
                odoo_row['Product Type'] = 'Storable Product'
                odoo_row['Tracking'] = 'By QTY'
                odoo_row['Product Category'] = row.get(COL_CAT2, '') 
                odoo_row['Invoicing policy'] = 'Ordered Quantities'
                
                first_row = False 
                
            final_odoo_rows.append(odoo_row)

df_products = pd.DataFrame(final_odoo_rows, columns=odoo_columns)

# 5. GENERATE ODOO ATTRIBUTES DEFINITION TAB
# ------------------------------------------------------------------------------
attributes_data = []
all_attrs = {
    'SIZE': df['Unified_Size'].dropna().unique(),
    'COLOR': df['Unified_Color'].dropna().unique(),
    'WEIGHT': df['Unified_Weight'].dropna().unique(),
    'CAPACITY': df['Unified_Capacity'].dropna().unique(),
    'DIMENSION': df['Unified_Dimension'].dropna().unique()
}

for attr_name, attr_values in all_attrs.items():
    if len(attr_values) > 0:
        sorted_vals = sorted([str(x) for x in attr_values])
        for i, val in enumerate(sorted_vals):
            attributes_data.append({
                'Attributes': attr_name if i == 0 else '', 
                'Values': val
            })

df_attributes = pd.DataFrame(attributes_data, columns=['Attributes', 'Values'])

# 6. EXPORT TO EXCEL
# ------------------------------------------------------------------------------
with pd.ExcelWriter('FINAL_Odoo_Import_Ready.xlsx', engine='openpyxl') as writer:
    df_products.to_excel(writer, sheet_name='Template for products', index=False)
    df_attributes.to_excel(writer, sheet_name='Product Attributes', index=False)

print("SUCCESS! Data Pipeline execution complete.")